In [1]:
import os
import pandas as pd
import re
import requests
import time
from bs4 import BeautifulSoup
from datetime import timedelta
from math import ceil
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
from openpyxl.workbook import Workbook
from requests.exceptions import ConnectTimeout, ReadTimeout

base_url = "https://www.tinko.ru"

In [2]:
def get_product_detail(url):
    start_time = time.time()

    retries, delay = 3, 2
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=10)
            if response.status_code != 200:
                print(f"Ошибка: {response} {url}")
                return {}
            soup = BeautifulSoup(response.text, "html.parser")
            break
        except (ConnectTimeout, ReadTimeout) as e:
            if attempt < retries - 1:
                time.sleep(delay * (attempt + 1))

    result_name = soup.find("h1").text
    detail_row_1 = soup.find("div", class_="product-detail__row-1")
    result_type = detail_row_1.find("h2").text

    detail_row_2 = soup.find("div", class_="product-detail__row-2")
    detail_prices = detail_row_2.find_all("div", class_="product-detail__price-wrapper")
    result_retail_price = detail_prices[0].find("span", class_="product-detail__price-value").text
    result_wholesale_price = result_retail_price
    if len(detail_prices) > 1 and "product-detail__price-light" in detail_prices[1].get("class", []):
        result_wholesale_price = detail_prices[1].find("span", class_="product-detail__price-value").text

    detail_properties = detail_row_1.find_all("div", class_="product-detail__property")
    result_code = detail_properties[0].find("span", class_="product-detail__property-code").text
    result_manufacturer = detail_properties[1].find("a").text
    article_tag = detail_properties[2].find("span", class_="product-detail__property-code")
    if article_tag:
        result_article = article_tag.text
    else:
        result_article = None

    result = {
        "name": result_name,
        "type": result_type,
        "retail_price": result_retail_price,
        "wholesale_price": result_wholesale_price,
        "code": result_code,
        "manufacturer": result_manufacturer,
        "article": result_article,
    }

    end_time = time.time()
    print(f"Получена информация о товаре: {url} ({ceil(end_time - start_time)}s)")

    return result

def get_products_detail_from_page(url):
    response = requests.get(url, timeout=10)
    if response.status_code != 200:
        print(f"Ошибка: {response} {url}")
        return pd.DataFrame()
    soup = BeautifulSoup(response.text, "html.parser")

    print(f"Извлечение данных о товарах на странице: {url}")
    result_category = re.sub(r"\s*\(\d+\)$", "", soup.find("h1").text)

    rows = []
    products_catalog = soup.find("div", class_="catalog-products")
    for product in products_catalog.find_all("div", class_="catalog-product"):
        product_title = product.find("p", class_="catalog-product__title")
        product_url = base_url + product_title.find("a").get("href")
        product_detail = get_product_detail(product_url)
        if product_detail:
            rows.append({
                "name": product_detail["name"],
                "type": product_detail["type"],
                "retail_price": product_detail["retail_price"],
                "wholesale_price": product_detail["wholesale_price"],
                "code": product_detail["code"],
                "manufacturer": product_detail["manufacturer"],
                "article": product_detail["article"],
                "category": result_category,
                "category_url": url,
                "url": product_url,
            })
        time.sleep(0.5)

    return pd.DataFrame(rows)

def get_products_detail_from_categories(start, end):
    products = pd.DataFrame(columns=[
        "name", "type", "retail_price", "wholesale_price", "code",
        "manufacturer", "article", "category", "category_url", "url",
    ])
    start_time = time.time()
    for i in range(start, end + 1):
        new_products = get_products_detail_from_page(f"{base_url}/catalog/category/{i}/")
        if not new_products.empty:
            products = pd.concat([products, new_products], ignore_index=True)
        time.sleep(1)
    end_time = time.time()
    elapsed_time = str(timedelta(seconds=round(end_time - start_time)))
    print(f"Загружены данные о {products.shape[0]} товарах за {elapsed_time}")
    return products

In [3]:
products = get_products_detail_from_categories(start=1, end=45)

Извлечение данных о товарах на странице: https://www.tinko.ru/catalog/category/1/
Получена информация о товаре: https://www.tinko.ru/catalog/product/303232/ (1s)
Получена информация о товаре: https://www.tinko.ru/catalog/product/248771/ (1s)
Получена информация о товаре: https://www.tinko.ru/catalog/product/004432/ (2s)
Получена информация о товаре: https://www.tinko.ru/catalog/product/004155/ (1s)
Получена информация о товаре: https://www.tinko.ru/catalog/product/246551/ (1s)
Получена информация о товаре: https://www.tinko.ru/catalog/product/004340/ (1s)
Получена информация о товаре: https://www.tinko.ru/catalog/product/302316/ (1s)
Получена информация о товаре: https://www.tinko.ru/catalog/product/004196/ (1s)
Получена информация о товаре: https://www.tinko.ru/catalog/product/224799/ (1s)
Получена информация о товаре: https://www.tinko.ru/catalog/product/209593/ (1s)
Получена информация о товаре: https://www.tinko.ru/catalog/product/248772/ (1s)
Получена информация о товаре: https://

In [4]:
products

,name,type,retail_price,wholesale_price,code,manufacturer,article,category,category_url,url
0,Сириус,Прибор приемно-контрольный и управления пожарный,"36 159,53","33 378,03",303232,Болид,1-520-887-052,Средства и системы охранно-пожарной сигнализации,https://www.tinko.ru/catalog/category/1/,https://www.tinko.ru/catalog/product/303232/
1,ДИП-34А-03 (ИП 212-34А),Извещатель пожарный дымовой оптико-электронный...,"1 520,97","1 403,98",248771,Болид,711-608-757,Средства и системы охранно-пожарной сигнализации,https://www.tinko.ru/catalog/category/1/,https://www.tinko.ru/catalog/product/248771/
2,С2000-М,Пульт контроля и управления,"12 721,31","11 742,74",004432,Болид,110-058-274,Средства и системы охранно-пожарной сигнализации,https://www.tinko.ru/catalog/category/1/,https://www.tinko.ru/catalog/product/004432/
3,С2000-КДЛ,Контроллер двухпроводной линии связи,"4 277,44","3 948,41",004155,Болид,10-468-001,Средства и системы охранно-пожарной сигнализации,https://www.tinko.ru/catalog/category/1/,https://www.tinko.ru/catalog/product/004155/
4,ДИП-34А-04 (ИП 212-34А),Извещатель пожарный дымовой оптико-электронный...,"2 014,22","1 859,28",246551,Болид,719-336-940,Средства и системы охранно-пожарной сигнализации,https://www.tinko.ru/catalog/category/1/,https://www.tinko.ru/catalog/product/246551/
...,...,...,...,...,...,...,...,...,...,...
431,"ИР 513-10 исп.1 ""АВАРИЙНЫЙ ВЫХОД"" (зелёный)",Устройство разблокировки двери,"1 299,00","1 284,00",251022,Рубеж,Rbz-123154,Извещатели пожарные ручные,https://www.tinko.ru/catalog/category/45/,https://www.tinko.ru/catalog/product/251022/
432,ИПР-55К (красный),Извещатель пожарный ручной,"449,00","364,65",267632,ИП Раченков А.В.,0000189,Извещатели пожарные ручные,https://www.tinko.ru/catalog/category/45/,https://www.tinko.ru/catalog/product/267632/
433,"УДП 535-26 ""Север"" ""Пуск Пожаротушения"", новый...",Устройство дистанционного пуска,"6 254,94","6 254,94",332733,Магнито-Контакт,09-0005-0000,Извещатели пожарные ручные,https://www.tinko.ru/catalog/category/45/,https://www.tinko.ru/catalog/product/332733/
434,"УДП 535-50 ""Север"" ""Пуск Пожаротушения"", без к...",Устройство дистанционного пуска,"6 254,94","6 254,94",279252,Магнито-Контакт,09-0006-0000,Извещатели пожарные ручные,https://www.tinko.ru/catalog/category/45/,https://www.tinko.ru/catalog/product/279252/


In [5]:
workbook = Workbook()
worksheet = workbook.active

header = products.columns.to_list()
worksheet.append(header)
for _, row in products.iterrows():
    worksheet.append(row.to_list())

for i in range(1, len(header) + 1):
    worksheet.column_dimensions[get_column_letter(i)].width = 16

for cell in worksheet[1]:
    cell.font = Font(bold=True)

for row in worksheet.iter_rows():
    for cell in row:
        cell.alignment = Alignment(wrap_text=True)

if os.path.exists("products.xlsx"):
    os.remove("products.xlsx")

workbook.save("products.xlsx")

In [6]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 436 entries, 0 to 435
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   name             436 non-null    object
 1   type             436 non-null    object
 2   retail_price     436 non-null    object
 3   wholesale_price  436 non-null    object
 4   code             436 non-null    object
 5   manufacturer     436 non-null    object
 6   article          227 non-null    object
 7   category         436 non-null    object
 8   category_url     436 non-null    object
 9   url              436 non-null    object
dtypes: object(10)
memory usage: 34.2+ KB


In [7]:
products.describe()

,name,type,retail_price,wholesale_price,code,manufacturer,article,category,category_url,url
count,436,436,436,436,436,436,227,436,436,436
unique,358,140,318,318,358,43,187,39,39,358
top,Астра-8,Извещатель пожарный дымовой оптико-электронный...,"9 150,00","9 150,00",002048,Риэлта,Rbz-338653,Средства и системы охранно-пожарной сигнализации,https://www.tinko.ru/catalog/category/1/,https://www.tinko.ru/catalog/product/002048/
freq,4,32,4,4,4,57,4,12,12,4


In [8]:
russian_letters = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюяАБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ'
english_letters = 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'

def classify_language(text):
    text = re.sub(r"[^a-zA-Zа-яА-ЯёЁ]", "", text)
    
    has_russian = any(c in russian_letters for c in text)
    has_english = any(c in english_letters for c in text)
    
    if has_russian and has_english:
        return "rus+eng"
    elif has_russian:
        return "rus"
    elif has_english:
        return "eng"
    else:
        return "none"

for column in ["name", "type", "manufacturer", "category"]:
    counts = products[column].apply(classify_language).value_counts()
    print(f"{counts}\n")

name
rus        327
rus+eng     68
eng         38
none         3
Name: count, dtype: int64

type
rus        431
rus+eng      5
Name: count, dtype: int64

manufacturer
rus    395
eng     41
Name: count, dtype: int64

category
rus        431
rus+eng      5
Name: count, dtype: int64

